# NB04 v2 — IDF-weighted lexical blocking

This is the corrected S4 experiment. It uses shared normalized name/address tokens, country partitions, IDF weighting, and a top-1% common-token filter. It reports recall at several K values and does not create a leaderboard submission.

In [ ]:
import json, math, os, pathlib, sys, time
import pandas as pd
ON_SAGEMAKER = pathlib.Path('/opt/ml').exists() or 'SM_CURRENT_HOST' in os.environ
ROOT = pathlib.Path(os.environ.get('BER_ROOT', '/home/ec2-user/SageMaker/ber' if ON_SAGEMAKER else '/Users/pocketfm/Desktop/amazon_ML/Working Folder'))
DATA, ART = ROOT / 'student_resource/dataset', ROOT / 'artifacts'
ART.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / 'src'))
from business_entity_resolution.blocking_stream import parquet_glob, write_index_parts
from business_entity_resolution.normalization_mined import load_mined_tables
mined = load_mined_tables(ART / 'normalization_v1.json')
index_root = ART / 'nb04_index_v1'
print(f'ROOT={ROOT} index={index_root}')

In [ ]:
t0 = time.time()
counts = {}
for name in ('train_source1', 'train_source2', 'train_source3'):
    rows, parts = write_index_parts(DATA / 'train' / f'{name}.tsv', index_root / name, mined=mined, chunksize=100_000)
    counts[name] = {'rows': rows, 'parts': parts}
    print(f'{name}: rows={rows:,} parts={parts} elapsed={time.time()-t0:.1f}s')

In [ ]:
import duckdb
con = duckdb.connect(database=str(ART / 'nb04_v2.duckdb'))
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='4GB'")
con.execute("PRAGMA max_temp_directory_size='6GB'")
(ART / 'nb04_v2_tmp').mkdir(parents=True, exist_ok=True)
spill = str(ART / 'nb04_v2_tmp').replace('\\', '\\\\').replace("'", "''")
con.execute("SET temp_directory = '" + spill + "'")
s1, s2, s3 = [parquet_glob(index_root / n) for n in ('train_source1', 'train_source2', 'train_source3')]
con.execute("CREATE OR REPLACE TEMP TABLE holdout_ids AS SELECT entity_id FROM read_parquet(?)", [str(ART / 'holdout_s1.parquet')])
con.execute(f"""CREATE OR REPLACE TABLE retrieval_source AS
SELECT * FROM read_parquet('{s2}') UNION ALL SELECT * FROM read_parquet('{s3}')""")
con.execute(f"""CREATE OR REPLACE TABLE retrieval_tokens AS
SELECT entity_id, country, 'name' AS field, token
FROM retrieval_source, UNNEST(string_split(name_key, chr(31))) AS u(token)
WHERE token <> ''
UNION ALL
SELECT entity_id, country, 'address', token
FROM retrieval_source, UNNEST(string_split(address_key, chr(31))) AS u(token)
WHERE token <> ''""")
con.execute("CREATE OR REPLACE TABLE retrieval_country_counts AS SELECT country, field, COUNT(DISTINCT entity_id) AS n FROM retrieval_tokens GROUP BY country, field")
con.execute("CREATE OR REPLACE TABLE retrieval_token_stats AS SELECT country, field, token, COUNT(DISTINCT entity_id) AS df FROM retrieval_tokens GROUP BY country, field, token")
con.execute(f"""CREATE OR REPLACE TABLE holdout_refs AS
SELECT * FROM read_parquet('{s1}') WHERE entity_id IN (SELECT entity_id FROM holdout_ids)""")
con.execute("""CREATE OR REPLACE TABLE holdout_ref_tokens AS
SELECT entity_id AS source1_entity_id, country, 'name' AS field, token
FROM holdout_refs, UNNEST(string_split(name_key, chr(31))) AS u(token) WHERE token <> ''
UNION ALL
SELECT entity_id, country, 'address', token
FROM holdout_refs, UNNEST(string_split(address_key, chr(31))) AS u(token) WHERE token <> ''
""")
# Keep only genuinely rare tokens. This bounded fallback is required on 8 GiB.
con.execute("""CREATE OR REPLACE TABLE idf_hits AS
SELECT r.source1_entity_id, x.entity_id AS candidate_id,
       SUM(LN((cc.n + 1.0) / (ts.df + 1.0))) AS score
FROM holdout_ref_tokens r
JOIN retrieval_token_stats ts USING (country, field, token)
JOIN retrieval_country_counts cc USING (country, field)
JOIN retrieval_tokens x USING (country, field, token)
WHERE ts.df <= 100 AND LENGTH(token) >= 2
GROUP BY r.source1_entity_id, x.entity_id
""")
con.execute("""CREATE OR REPLACE TABLE ranked_candidates AS
SELECT source1_entity_id, candidate_id, score,
       ROW_NUMBER() OVER (PARTITION BY source1_entity_id ORDER BY score DESC, candidate_id) AS rank
FROM idf_hits
""")
candidate_stats = con.execute("SELECT COUNT(*) AS rows, COUNT(DISTINCT source1_entity_id) AS s1, AVG(n) AS avg_per_s1 FROM (SELECT source1_entity_id, COUNT(*) n FROM ranked_candidates GROUP BY source1_entity_id)").fetchone()
print({'ranked_rows': int(candidate_stats[0]), 's1_with_candidates': int(candidate_stats[1]), 'avg_candidates_per_s1': float(candidate_stats[2] or 0)})

In [ ]:
gt = pd.read_csv(DATA / 'train/train_ground_truth.tsv', sep='\t', dtype='string', keep_default_na=False)
holdout_set = set(pd.read_parquet(ART / 'holdout_s1.parquet', columns=['entity_id'])['entity_id'])
gt = gt[gt.source1_entity_id.isin(holdout_set)].copy()
gt['matched_entity_ids'] = gt['matched_entity_ids'].fillna('')
positive = gt.assign(candidate_id=gt.matched_entity_ids.str.split(',')).explode('candidate_id')
positive = positive[positive.candidate_id.ne('')][['source1_entity_id', 'candidate_id']]
con.register('holdout_positive', positive)
results = []
for k in (10, 20, 30, 50, 100):
    hit = con.execute(f"""SELECT COUNT(*) AS positives, SUM(CASE WHEN c.candidate_id IS NOT NULL THEN 1 ELSE 0 END) AS retained
    FROM holdout_positive p LEFT JOIN (SELECT DISTINCT source1_entity_id, candidate_id FROM ranked_candidates WHERE rank <= {k}) c
      ON p.source1_entity_id=c.source1_entity_id AND p.candidate_id=c.candidate_id""").fetchone()
    results.append({'k': k, 'positive_pairs': int(hit[0]), 'retained': int(hit[1]), 'recall': float(hit[1] / max(1, hit[0]))})
result = {'stage': 'S4-v3-rare-token', 'counts': counts, 'candidate_stats': {'ranked_rows': int(candidate_stats[0]), 's1_with_candidates': int(candidate_stats[1]), 'avg_candidates_per_s1': float(candidate_stats[2] or 0)}, 'recall_at_k': results, 'config': {'common_token_filter': 'df <= 100 per country/field', 'fields': ['name', 'address'], 'idf': True, 'max_temp_directory_size': '6GB'}}
(ART / 'nb04_v2_recall_summary.json').write_text(json.dumps(result, indent=2) + '\n')
print(json.dumps(result, indent=2))